In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

In [3]:
class backbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Remove avgpool and fc — keep only the feature extractor
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        # ResNet18 outputs (512, 14, 14) for 448x448 input — pool down to 7x7
        self.adapt = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adapt(x)
        return x  # (batch, 512, 7, 7)


class head(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(512, 1024, 3, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 1024, 3, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1),
            nn.Flatten(),
            nn.Linear(1024 * 7 * 7, 4096),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.5),
            nn.Linear(4096, 7 * 7 * 50)
        )

    def forward(self, x):
        return self.features(x)


class YOLO(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = backbone()
        self.head = head()

    def forward(self, x):
        x = self.backbone(x)
        x = self.head(x)
        x = x.view(-1, 7, 7, 50)
        return x

In [4]:
def iou(box1, box2):
    x1 = torch.max(box1[..., 0] - box1[..., 2]/2, box2[..., 0] - box2[..., 2]/2)
    y1 = torch.max(box1[..., 1] - box1[..., 3]/2, box2[..., 1] - box2[..., 3]/2)
    x2 = torch.min(box1[..., 0] + box1[..., 2]/2, box2[..., 0] + box2[..., 2]/2)
    y2 = torch.min(box1[..., 1] + box1[..., 3]/2, box2[..., 1] + box2[..., 3]/2)
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    box1_area = box1[..., 2] * box1[..., 3]
    box2_area = box2[..., 2] * box2[..., 3]
    union = box1_area + box2_area - intersection
    return intersection / (union + 1e-6)


class YOLOLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss(reduction='sum')
        self.lambda_coord = 5
        self.lambda_noobj = 0.5

    def forward(self, predictions, targets):
        pred_box1 = predictions[..., 0:4]
        pred_box2 = predictions[..., 5:9]
        target_box = targets[..., 0:4]
        pred_conf1 = predictions[..., 4:5]
        pred_conf2 = predictions[..., 9:10]
        obj   = targets[..., 4:5]
        noobj = 1 - obj

        iou_box1 = iou(pred_box1, target_box)
        iou_box2 = iou(pred_box2, target_box)
        best_box = (iou_box2 > iou_box1).float().unsqueeze(-1)
        pred_box = (1 - best_box) * pred_box1 + best_box * pred_box2

        coord_loss = self.mse(
            obj * pred_box[..., 0:2],
            obj * target_box[..., 0:2]
        )
        size_loss = self.mse(
            obj * torch.sign(pred_box[..., 2:4]) * torch.sqrt(torch.abs(pred_box[..., 2:4]) + 1e-6),
            obj * torch.sqrt(target_box[..., 2:4] + 1e-6)
        )
        pred_conf = (1 - best_box) * pred_conf1 + best_box * pred_conf2
        conf_loss_obj   = self.mse(obj   * pred_conf,  obj   * targets[..., 4:5])
        conf_loss_noobj = self.mse(noobj * pred_conf1, noobj * targets[..., 4:5]) + \
                          self.mse(noobj * pred_conf2, noobj * targets[..., 4:5])
        class_loss = self.mse(
            obj * predictions[..., 10:],
            obj * targets[..., 10:]
        )
        return (self.lambda_coord * (coord_loss + size_loss)
                + conf_loss_obj
                + self.lambda_noobj * conf_loss_noobj
                + class_loss)

In [5]:
def encode_target(annotation, grid_size=7):
    target = torch.zeros(grid_size, grid_size, 50)
    size = annotation['annotation']['size']
    img_w = float(size['width'])
    img_h = float(size['height'])

    for obj in annotation['annotation']['object']:
        class_name = obj['name']
        if class_name not in VOC_CLASSES:
            continue
        class_idx = VOC_CLASSES.index(class_name)
        xmin = float(obj['bndbox']['xmin']) / img_w
        ymin = float(obj['bndbox']['ymin']) / img_h
        xmax = float(obj['bndbox']['xmax']) / img_w
        ymax = float(obj['bndbox']['ymax']) / img_h

        x_center = (xmin + xmax) / 2
        y_center = (ymin + ymax) / 2
        width    = xmax - xmin
        height   = ymax - ymin

        col = min(int(x_center * grid_size), grid_size - 1)
        row = min(int(y_center * grid_size), grid_size - 1)
        x_cell = x_center * grid_size - col
        y_cell = y_center * grid_size - row

        if target[row, col, 4] == 0:
            target[row, col, 0:5] = torch.tensor([x_cell, y_cell, width, height, 1])
            target[row, col, 10 + class_idx] = 1

    return target


class VOCDataset(Dataset):
    def __init__(self, root, image_set='train', transform=None):
        self.transform = transform
        self.dataset = datasets.VOCDetection(
            root=root, year='2012',
            image_set=image_set, download=True
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, annotation = self.dataset[idx]
        image = image.resize((448, 448))
        if self.transform:
            image = self.transform(image)
        target = encode_target(annotation)
        return image, target

In [6]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomCrop(448, padding=20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = VOCDataset(root='data', image_set='train', transform=train_transform)
val_dataset   = VOCDataset(root='data', image_set='val',   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

100%|██████████| 2.00G/2.00G [00:44<00:00, 45.1MB/s]


In [7]:
model = YOLO()
model = nn.DataParallel(model)
model = model.to(device)
loss_fn = YOLOLoss()

# Start with backbone frozen — let the head stabilize first
for param in model.module.backbone.features.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=5e-4
)

num_epochs   = 150
scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
best_val_loss    = float('inf')
patience         = 20
epochs_no_improve = 0
backbone_unfrozen = False

for epoch in range(num_epochs):
    current_epoch = epoch + 1

    # Unfreeze backbone after epoch 10
    if current_epoch == 11 and not backbone_unfrozen:
        print("Unfreezing backbone...")
        for param in model.module.backbone.features.parameters():
            param.requires_grad = True
        # Rebuild optimizer to include backbone params, with lower LR for backbone
        optimizer = torch.optim.Adam([
            {'params': model.module.backbone.parameters(), 'lr': 1e-5},
            {'params': model.module.head.parameters(),     'lr': 1e-4}
        ], weight_decay=5e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - 10)
        backbone_unfrozen = True

    # Training
    model.train()
    total_train_loss = 0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    # Validation every 5 epochs
    if current_epoch % 5 == 0:
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(device), targets.to(device)
                outputs = model(images)
                loss = loss_fn(outputs, targets)
                total_val_loss += loss.item()

        avg_train = total_train_loss / len(train_loader)
        avg_val   = total_val_loss   / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{current_epoch}/{num_epochs}] | Train: {avg_train:.4f} | Val: {avg_val:.4f} | LR: {current_lr:.7f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'yolo_resnet_best.pth')
            print(f"  ✓ Best model saved (val loss {avg_val:.4f})")
        else:
            epochs_no_improve += 5
            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {current_epoch}")
                break

    if current_epoch % 10 == 0:
        torch.save(model.state_dict(), f'yolo_resnet_epoch_{current_epoch}.pth')
        print(f"  Checkpoint saved at epoch {current_epoch}")

    scheduler.step()

print("Training complete!")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 195MB/s]


Epoch [5/150] | Train: 253.5782 | Val: 218.5453 | LR: 0.0000998
  ✓ Best model saved (val loss 218.5453)
Epoch [10/150] | Train: 203.8035 | Val: 190.5267 | LR: 0.0000991
  ✓ Best model saved (val loss 190.5267)
  Checkpoint saved at epoch 10
Unfreezing backbone...
Epoch [15/150] | Train: 172.8341 | Val: 198.6388 | LR: 0.0000100
Epoch [20/150] | Train: 152.4160 | Val: 164.3367 | LR: 0.0000099
  ✓ Best model saved (val loss 164.3367)
  Checkpoint saved at epoch 20
Epoch [25/150] | Train: 137.5988 | Val: 161.8925 | LR: 0.0000098
  ✓ Best model saved (val loss 161.8925)
Epoch [30/150] | Train: 126.2397 | Val: 160.5232 | LR: 0.0000096
  ✓ Best model saved (val loss 160.5232)
  Checkpoint saved at epoch 30
Epoch [35/150] | Train: 116.1216 | Val: 156.3404 | LR: 0.0000093
  ✓ Best model saved (val loss 156.3404)
Epoch [40/150] | Train: 107.4808 | Val: 155.1727 | LR: 0.0000090
  ✓ Best model saved (val loss 155.1727)
  Checkpoint saved at epoch 40
Epoch [45/150] | Train: 100.6930 | Val: 155.460